In [ ]:
# Cell: Gắn Google Drive vào Colab để notebook đọc/ghi dữ liệu trong thư mục OBAD.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell: Chuyển vào thư mục source của mô hình L1 TCN trước khi chạy các script L1.
%cd /content/drive/MyDrive/OBAD/modeling/l1_tcn/src

In [ ]:
# Cell: Kiểm tra GPU/CUDA để biết Colab đang chạy bằng GPU hay CPU.
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# Cell: Cài các thư viện Python tối thiểu cần cho phần L1.
!pip install -q pandas numpy pyyaml

In [ ]:
# Cell: Kiểm tra nhanh các đường dẫn quan trọng trong cấu hình L1.
!python -c "from config import load_yaml, build_paths; cfg=load_yaml('../configs/base.yaml'); p=build_paths(cfg, '../configs/base.yaml'); print(p.l1_full); print(p.lenient_train); print(p.strict_train)"

In [ ]:
# Cell: Chạy thử training L1 profile lenient với giới hạn dữ liệu để kiểm tra pipeline.
!python train.py --config ../configs/base.yaml --profile lenient --limit-train-windows 200000

In [ ]:
# Cell: Chạy thử training L1 profile strict với giới hạn dữ liệu để kiểm tra pipeline.
!python train.py --config ../configs/base.yaml --profile strict --limit-train-windows 200000

In [ ]:
# Cell: Train đầy đủ mô hình L1 profile lenient.
!python train.py --config ../configs/base.yaml --profile lenient

In [ ]:
# Cell: Train đầy đủ mô hình L1 profile strict.
!python train.py --config ../configs/base.yaml --profile strict

In [ ]:
# Cell: Liệt kê artifact L1 sau train để xác nhận file đầu ra đã được tạo.
!ls -lh /content/drive/MyDrive/OBAD/modeling/l1_tcn/artifacts/lenient
!ls -lh /content/drive/MyDrive/OBAD/modeling/l1_tcn/artifacts/strict

In [ ]:
# Cell: Đọc summary L1 valid/test để kiểm tra tỷ lệ anomaly của từng profile.
import json

for profile in ["lenient", "strict"]:
    print("\n====", profile, "valid ====")
    path = f"/content/drive/MyDrive/OBAD/modeling/l1_tcn/artifacts/{profile}/valid_anomaly_summary.json"
    data = json.load(open(path, "r", encoding="utf-8"))
    print("total_windows:", data["total_windows"])
    print("anomaly_windows:", data["anomaly_windows"])
    print("anomaly_rate:", data["anomaly_rate"])

    print("\n====", profile, "test ====")
    path = f"/content/drive/MyDrive/OBAD/modeling/l1_tcn/artifacts/{profile}/test_anomaly_summary.json"
    data = json.load(open(path, "r", encoding="utf-8"))
    print("total_windows:", data["total_windows"])
    print("anomaly_windows:", data["anomaly_windows"])
    print("anomaly_rate:", data["anomaly_rate"])

In [ ]:
# Cell: Score toàn bộ dữ liệu bằng mô hình L1 đã train.
!python score_full_l1.py --config ../configs/base.yaml

In [ ]:
# Cell: Xem nhanh kết quả score L1 thô và phân bố behavior anomaly theo máy.
import pandas as pd

path = "/content/drive/MyDrive/OBAD/data/dataModel/l1/scored/ai_l1_operation_anomaly_result.csv"
df = pd.read_csv(path)

print(df.shape)
print(df.head())
print(df["behavior_reason"].value_counts(dropna=False))
print(df.groupby("machine_id")["is_behavior_anomaly"].mean().sort_values(ascending=False))

In [ ]:
# Cell: Rebuild quyết định L1 production từ kết quả score thô.
!python rebuild_l1_final_decision.py

In [ ]:
# Cell: Kiểm tra file L1 production sau khi rebuild.
import pandas as pd

path = "/content/drive/MyDrive/OBAD/data/dataModel/l1/scored/ai_l1_operation_anomaly_result_production.csv"
df = pd.read_csv(path)

print(df.shape)
print(df.head())
print(df["behavior_reason"].value_counts(dropna=False))
print(df.groupby("machine_id")["is_behavior_anomaly"].mean().sort_values(ascending=False))

In [ ]:
# Cell: Ghép tín hiệu L1 vào dataset L2.
%cd /content/drive/MyDrive/OBAD/modeling/l2_fault_classifier/src
!python join_l1_score_to_l2.py --config ../configs/base.yaml

In [ ]:
# Cell: Kiểm tra tỷ lệ join thiếu và các tín hiệu L1 đã ghép theo từng split.
import pandas as pd

base = "/content/drive/MyDrive/OBAD/data/dataModel/l2"

for split in ["train", "valid", "test"]:
    path = f"{base}/{split}_with_l1_score.csv"
    df = pd.read_csv(path)

    print("\n===", split, "===")
    print(df.shape)
    print(df[[
        "l1_join_missing_flag",
        "l1_score_available_flag",
        "is_behavior_anomaly",
        "is_sensitive_warning",
        "behavior_anomaly_score",
        "behavior_sensitive_score",
    ]].mean(numeric_only=True))

In [ ]:
# Cell: Đọc các report sau khi join L1 vào L2.
import pandas as pd

report = "/content/drive/MyDrive/OBAD/data/dataModel/l2/with_l1_report"

print(pd.read_csv(f"{report}/join_l1_to_l2_summary.csv"))
print(pd.read_csv(f"{report}/l2_target_distribution_with_l1.csv"))

In [ ]:
# Cell: Xem top dòng tín hiệu L1 theo target L2 để đánh giá độ liên quan.
sig = pd.read_csv(f"{report}/l1_signal_by_l2_target.csv")
sig.head(20)

In [ ]:
# Cell: Chuẩn bị feature L2 theo feature_policy, bao gồm cân bằng lại tín hiệu strict/lenient.
!python prepare_l2_features.py --config ../configs/feature_policy.yaml


In [ ]:
# Cell: Kiểm tra file L2 ready và policy feature sau bước chuẩn bị dữ liệu.
import pandas as pd, json

base = "/content/drive/MyDrive/OBAD/data/dataModel/l2/prepared"
report = "/content/drive/MyDrive/OBAD/data/dataModel/l2/prepared_report"

for split in ["train", "valid", "test"]:
    df = pd.read_csv(f"{base}/{split}_l2_ready.csv", nrows=5)
    print("\n===", split, "===")
    print(df.shape)
    print([c for c in df.columns if c.startswith("l1_")][:30])

policy = json.load(open(f"{report}/l2_feature_policy.json", "r", encoding="utf-8"))
print(policy["profile_sizes"])


=== train ===
(5, 84)
['l1_model_version', 'l1_decision_policy', 'l1_join_missing_flag', 'l1_score_available_flag', 'l1_lenient_norm_clip', 'l1_lenient_norm_log', 'l1_strict_norm_clip', 'l1_strict_norm_log', 'l1_behavior_anomaly_score_clip', 'l1_behavior_anomaly_score_log', 'l1_behavior_sensitive_score_clip', 'l1_behavior_sensitive_score_log', 'l1_behavior_combined_score_clip', 'l1_behavior_combined_score_log', 'l1_score_lenient_clip', 'l1_score_lenient_log', 'l1_score_strict_clip', 'l1_score_strict_log', 'l1_strict_lenient_gap_log', 'l1_strict_lenient_ratio_log', 'l1_score_balance_index', 'l1_behavior_anomaly_flag']

=== valid ===
(5, 84)
['l1_model_version', 'l1_decision_policy', 'l1_join_missing_flag', 'l1_score_available_flag', 'l1_lenient_norm_clip', 'l1_lenient_norm_log', 'l1_strict_norm_clip', 'l1_strict_norm_log', 'l1_behavior_anomaly_score_clip', 'l1_behavior_anomaly_score_log', 'l1_behavior_sensitive_score_clip', 'l1_behavior_sensitive_score_log', 'l1_behavior_combined_score

In [ ]:
# Cell: Chuyển vào source L2 và cài thư viện train L2 bằng LightGBM.
%cd /content/drive/MyDrive/OBAD/modeling/l2_fault_classifier/src
!pip install -q lightgbm scikit-learn joblib pyyaml

/content/drive/MyDrive/OBAD/modeling/l2_fault_classifier/src


In [ ]:
# Cell: Chạy thử train L2 trên CPU với một target và giới hạn dòng để kiểm tra pipeline.
#test train bằng CPU
!python train_l2_multilabel.py \
  --config ../configs/train_l2.yaml \
  --profiles safe \
  --targets future_fault_within_30min \
  --max-train-rows 200000

In [ ]:
# Cell: Train đầy đủ L2 multi-label theo cấu hình train_l2.yaml.
!python train_l2_multilabel.py --config ../configs/train_l2.yaml

In [ ]:
# Cell: Train thử L2 bằng backend XGBoost cho các profile được chọn.
!pip install -q xgboost

!python train_l2_multilabel.py \
  --config ../configs/train_l2.yaml \
  --backend xgboost \
  --max-train-rows 200000 \
  --profiles safe,strict_continuous


In [ ]:
# Cell: Đọc report training mới nhất và cấu hình chọn profile production.
import pandas as pd, json, glob

report_root = "/content/drive/MyDrive/OBAD/data/dataModel/l2/model_report"
latest = sorted(glob.glob(report_root + "/*"))[-1]
print(latest)

summary = pd.read_csv(f"{latest}/l2_training_summary.csv")
print(summary[[
    "profile",
    "target",
    "valid_average_precision",
    "valid_roc_auc",
    "valid_threshold_f1",
    "test_average_precision",
    "test_roc_auc",
    "test_threshold_f1"
]].sort_values(["target", "valid_average_precision"], ascending=[True, False]))

selection = json.load(open(f"{latest}/production_profile_selection.json", "r", encoding="utf-8"))
print(json.dumps(selection, ensure_ascii=False, indent=2))

In [3]:
# Cell: Cài thư viện cần cho bước score L2 production.
%cd /content/drive/MyDrive/OBAD/modeling/l2_fault_classifier/src

!pip install -q lightgbm xgboost scikit-learn joblib pyyaml

/content/drive/MyDrive/OBAD/modeling/l2_fault_classifier/src


In [21]:
# Cell: Tạo cấu hình score local để ghi output tạm vào /content, tránh ghi chậm trực tiếp lên Drive.
from pathlib import Path
import yaml

src = Path("/content/drive/MyDrive/OBAD/modeling/l2_fault_classifier/configs/score_l2.yaml")

cfg = yaml.safe_load(src.read_text(encoding="utf-8"))

cfg["paths"]["output"]["root_dir"] = "/content/obad_l2_scored"
cfg["paths"]["output"]["report_root"] = "/content/obad_l2_production_report"

cfg["data"]["output_compression"] = None
cfg["data"]["chunksize"] = 150000

dst = Path("/content/drive/MyDrive/OBAD/modeling/l2_fault_classifier/configs/score_l2_local.yaml")
dst.write_text(
    yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False),
    encoding="utf-8"
)

print(dst)
print("output root:", cfg["paths"]["output"]["root_dir"])
print("report root:", cfg["paths"]["output"]["report_root"])
print("compression:", cfg["data"]["output_compression"])
print("chunksize:", cfg["data"]["chunksize"])

/content/drive/MyDrive/OBAD/modeling/l2_fault_classifier/configs/score_l2_local.yaml
output root: /content/obad_l2_scored
report root: /content/obad_l2_production_report
compression: None
chunksize: 150000


In [ ]:
# Cell: Kiểm tra lại các đường dẫn trong cấu hình score local.
from pathlib import Path
import yaml

p = Path("/content/drive/MyDrive/OBAD/modeling/l2_fault_classifier/configs/score_l2_local.yaml")
cfg = yaml.safe_load(p.read_text(encoding="utf-8"))

print("batch06 report_root:", cfg["paths"]["batch06"]["report_root"])
print("artifact_root:", cfg["paths"]["batch06"]["artifact_root"])
print("output root:", cfg["paths"]["output"]["root_dir"])
print("report output:", cfg["paths"]["output"]["report_root"])

In [22]:
# Cell: Score split valid bằng model L2 production và ghi output tạm.
%cd /content/drive/MyDrive/OBAD/modeling/l2_fault_classifier/src

!rm -rf /content/obad_l2_scored
!rm -rf /content/obad_l2_production_report

!python score_l2_production.py \
  --config ../configs/score_l2_local.yaml \
  --run-id l2_multilabel_20260711_043347 \
  --splits valid

/content/drive/MyDrive/OBAD/modeling/l2_fault_classifier/src
Batch 07 L2 production scoring
Run id: l2_multilabel_20260711_043347
Selected models:
  future_fault_within_10_events: profile=safe backend=lightgbm threshold=0.130113
  future_fault_within_30_events: profile=strict_continuous backend=lightgbm threshold=0.072355
  future_fault_within_30min: profile=safe backend=lightgbm threshold=0.070673
  future_fault_within_60min: profile=safe backend=lightgbm threshold=0.082103
  future_maintenance_within_30_events: profile=strict_continuous backend=lightgbm threshold=0.108843
  future_repair_within_30_events: profile=strict_continuous backend=lightgbm threshold=0.072070

[valid] score /content/drive/MyDrive/OBAD/data/dataModel/l2/prepared/valid_l2_ready.csv
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/skle

In [31]:
# Cell: Score split test bằng model L2 production.
%cd /content/drive/MyDrive/OBAD/modeling/l2_fault_classifier/src

!python score_l2_production.py \
  --config ../configs/score_l2_local.yaml \
  --run-id l2_multilabel_20260711_043347 \
  --splits test

/content/drive/MyDrive/OBAD/modeling/l2_fault_classifier/src
Batch 07 L2 production scoring
Run id: l2_multilabel_20260711_043347
Selected models:
  future_fault_within_10_events: profile=safe backend=lightgbm threshold=0.130113
  future_fault_within_30_events: profile=strict_continuous backend=lightgbm threshold=0.072355
  future_fault_within_30min: profile=safe backend=lightgbm threshold=0.070673
  future_fault_within_60min: profile=safe backend=lightgbm threshold=0.082103
  future_maintenance_within_30_events: profile=strict_continuous backend=lightgbm threshold=0.108843
  future_repair_within_30_events: profile=strict_continuous backend=lightgbm threshold=0.072070

[test] score /content/drive/MyDrive/OBAD/data/dataModel/l2/prepared/test_l2_ready.csv
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklear

In [34]:
# Cell: Score split train bằng model L2 production.
%cd /content/drive/MyDrive/OBAD/modeling/l2_fault_classifier/src

!python score_l2_production.py \
  --config ../configs/score_l2_local.yaml \
  --run-id l2_multilabel_20260711_043347 \
  --splits train

/content/drive/MyDrive/OBAD/modeling/l2_fault_classifier/src
Batch 07 L2 production scoring
Run id: l2_multilabel_20260711_043347
Selected models:
  future_fault_within_10_events: profile=safe backend=lightgbm threshold=0.130113
  future_fault_within_30_events: profile=strict_continuous backend=lightgbm threshold=0.072355
  future_fault_within_30min: profile=safe backend=lightgbm threshold=0.070673
  future_fault_within_60min: profile=safe backend=lightgbm threshold=0.082103
  future_maintenance_within_30_events: profile=strict_continuous backend=lightgbm threshold=0.108843
  future_repair_within_30_events: profile=strict_continuous backend=lightgbm threshold=0.072070

[train] score /content/drive/MyDrive/OBAD/data/dataModel/l2/prepared/train_l2_ready.csv
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/skle

In [23]:
# Cell: Kiểm tra output score L2 valid và các metric model đã chọn.
import pandas as pd
from pathlib import Path

run_id = "l2_multilabel_20260711_043347"

root = Path(f"/content/obad_l2_scored/{run_id}")
report = Path(f"/content/obad_l2_production_report/{run_id}")

df = pd.read_csv(root / "valid_l2_fault_judgment.csv")
metrics = pd.read_csv(report / "l2_selected_model_metrics.csv")

print("df shape:", df.shape)
print("columns:", df.columns.tolist())

print("\nACTION LEVEL count:")
print(df["action_level"].value_counts())

print("\nACTION LEVEL pct:")
print((df["action_level"].value_counts(normalize=True) * 100).round(3))

print("\nFAULT JUDGMENT count:")
print(df["fault_judgment"].value_counts().head(30))

print("\nFAULT JUDGMENT pct:")
print((df["fault_judgment"].value_counts(normalize=True).head(30) * 100).round(3))

print("\nMETRICS:")
print(metrics[[
    "split",
    "target",
    "profile",
    "average_precision",
    "roc_auc",
    "f1",
    "precision",
    "recall",
    "pred_positive_rate"
]])

df shape: (609268, 65)
columns: ['event_id', 'machine_id', 'sequence_segment_id', 'event_order_in_segment', 'status_id', 'status_type_code', 'current_signal_code', 'known_fault_status', 'known_maintenance_status', 'known_repair_status', 'off_with_fault_status', 'energy_inconsistency_flag', 'data_quality_issue_flag', 'data_quality_issue_count', 'time_quality_issue_flag', 'kwh_quality_issue_flag', 'fault_evidence_count', 'maintenance_evidence_count', 'is_behavior_anomaly', 'is_sensitive_warning', 'behavior_anomaly_score', 'behavior_sensitive_score', 'behavior_combined_score', 'l1_behavior_anomaly_score_log', 'l1_behavior_sensitive_score_log', 'l1_behavior_combined_score_log', 'l1_score_available_flag', 'l1_join_missing_flag', 'risk_fault_10_events', 'pred_fault_10_events', 'threshold_fault_10_events', 'profile_fault_10_events', 'risk_fault_30_events', 'pred_fault_30_events', 'threshold_fault_30_events', 'profile_fault_30_events', 'risk_fault_30min', 'pred_fault_30min', 'threshold_fault_3

In [24]:
# Cell: So sánh threshold lưu trong output với threshold F1 tốt nhất trên split valid.
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_curve, f1_score, precision_score, recall_score, average_precision_score, roc_auc_score

run_id = "l2_multilabel_20260711_043347"

pred_path = f"/content/obad_l2_scored/{run_id}/valid_l2_fault_judgment.csv"
label_path = "/content/drive/MyDrive/OBAD/data/dataModel/l2/prepared/valid_l2_ready.csv"

target_map = {
    "future_fault_within_10_events": "fault_10_events",
    "future_fault_within_30_events": "fault_30_events",
    "future_fault_within_30min": "fault_30min",
    "future_fault_within_60min": "fault_60min",
    "future_maintenance_within_30_events": "maintenance_30_events",
    "future_repair_within_30_events": "repair_30_events",
}

use_pred_cols = ["event_id"]
for name in target_map.values():
    use_pred_cols += [f"risk_{name}", f"pred_{name}", f"threshold_{name}"]

use_label_cols = ["event_id"] + list(target_map.keys())

pred = pd.read_csv(pred_path, usecols=use_pred_cols)
labels = pd.read_csv(label_path, usecols=use_label_cols)

m = pred.merge(labels, on="event_id", how="inner")
print("merged:", m.shape)

rows = []

for target, name in target_map.items():
    y = m[target].astype(int).to_numpy()
    p = m[f"risk_{name}"].astype(float).to_numpy()

    stored_thr = float(m[f"threshold_{name}"].dropna().iloc[0])
    yhat_stored = (p >= stored_thr).astype(int)

    precision, recall, thresholds = precision_recall_curve(y, p)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)

    if len(thresholds) > 0:
        best_idx = int(np.nanargmax(f1[:-1]))
        best_thr = float(thresholds[best_idx])
        best_f1 = float(f1[best_idx])
        best_precision = float(precision[best_idx])
        best_recall = float(recall[best_idx])
    else:
        best_thr = 0.5
        best_f1 = 0.0
        best_precision = 0.0
        best_recall = 0.0

    rows.append({
        "target": target,
        "positive_rate": y.mean(),
        "stored_threshold": stored_thr,
        "stored_pred_rate": yhat_stored.mean(),
        "stored_precision": precision_score(y, yhat_stored, zero_division=0),
        "stored_recall": recall_score(y, yhat_stored, zero_division=0),
        "stored_f1": f1_score(y, yhat_stored, zero_division=0),
        "best_f1_threshold": best_thr,
        "best_f1": best_f1,
        "best_precision": best_precision,
        "best_recall": best_recall,
        "average_precision": average_precision_score(y, p),
        "roc_auc": roc_auc_score(y, p),
    })

threshold_check = pd.DataFrame(rows)
print(threshold_check)

merged: (609268, 25)
                                target  positive_rate  stored_threshold  \
0        future_fault_within_10_events       0.008960          0.130113   
1        future_fault_within_30_events       0.020725          0.072355   
2            future_fault_within_30min       0.025531          0.070673   
3            future_fault_within_60min       0.042118          0.082103   
4  future_maintenance_within_30_events       0.027041          0.108843   
5       future_repair_within_30_events       0.020318          0.072070   

   stored_pred_rate  stored_precision  stored_recall  stored_f1  \
0          0.001837          0.726542       0.148928   0.247188   
1          0.018680          0.207012       0.186584   0.196268   
2          0.003718          0.232671       0.033880   0.059147   
3          0.018534          0.265675       0.116909   0.162368   
4          0.008852          0.644910       0.211108   0.318090   
5          0.018680          0.196556       0.18070

In [25]:
# Cell: Đánh giá phân bố action level và tỷ lệ target theo từng action.
import pandas as pd

run_id = "l2_multilabel_20260711_043347"

pred_path = f"/content/obad_l2_scored/{run_id}/valid_l2_fault_judgment.csv"
label_path = "/content/drive/MyDrive/OBAD/data/dataModel/l2/prepared/valid_l2_ready.csv"

targets = [
    "future_fault_within_10_events",
    "future_fault_within_30_events",
    "future_fault_within_30min",
    "future_fault_within_60min",
    "future_maintenance_within_30_events",
    "future_repair_within_30_events",
]

pred = pd.read_csv(pred_path)
labels = pd.read_csv(label_path, usecols=["event_id"] + targets)

m = pred.merge(labels, on="event_id", how="left")

action_count = m["action_level"].value_counts().rename("count")
action_pct = (m["action_level"].value_counts(normalize=True) * 100).rename("pct")

action_target_rate = m.groupby("action_level")[targets].mean()
action_target_sum = m.groupby("action_level")[targets].sum()

audit = pd.concat([action_count, action_pct], axis=1)
print("\nACTION DISTRIBUTION:")
print(audit)

print("\nTARGET RATE BY ACTION:")
print((action_target_rate * 100).round(3))

print("\nTARGET COUNT BY ACTION:")
print(action_target_sum.astype(int))


ACTION DISTRIBUTION:
               count        pct
action_level                   
MONITOR       414475  68.028355
LOW           179797  29.510330
HIGH           10353   1.699252
MEDIUM          3400   0.558047
CRITICAL        1243   0.204015

TARGET RATE BY ACTION:
              future_fault_within_10_events  future_fault_within_30_events  \
action_level                                                                 
CRITICAL                             69.509                         71.762   
HIGH                                 10.152                         14.653   
LOW                                   0.760                          2.149   
MEDIUM                                3.618                          6.088   
MONITOR                               0.496                          1.483   

              future_fault_within_30min  future_fault_within_60min  \
action_level                                                         
CRITICAL                         71.923    

In [26]:
# Cell: Tính precision/recall top-k để xem model bắt được target ở nhóm rủi ro cao ra sao.
import numpy as np
import pandas as pd

run_id = "l2_multilabel_20260711_043347"

pred_path = f"/content/obad_l2_scored/{run_id}/valid_l2_fault_judgment.csv"
label_path = "/content/drive/MyDrive/OBAD/data/dataModel/l2/prepared/valid_l2_ready.csv"

target_map = {
    "future_fault_within_10_events": "fault_10_events",
    "future_fault_within_30_events": "fault_30_events",
    "future_fault_within_30min": "fault_30min",
    "future_fault_within_60min": "fault_60min",
    "future_maintenance_within_30_events": "maintenance_30_events",
    "future_repair_within_30_events": "repair_30_events",
}

pred = pd.read_csv(pred_path, usecols=["event_id"] + [f"risk_{v}" for v in target_map.values()])
labels = pd.read_csv(label_path, usecols=["event_id"] + list(target_map.keys()))
m = pred.merge(labels, on="event_id", how="inner")

rows = []

for target, name in target_map.items():
    risk_col = f"risk_{name}"
    tmp = m[[target, risk_col]].copy()
    tmp = tmp.sort_values(risk_col, ascending=False)

    total_pos = tmp[target].sum()
    n = len(tmp)

    for frac in [0.005, 0.01, 0.02, 0.05, 0.10]:
        k = max(1, int(np.ceil(n * frac)))
        top = tmp.head(k)
        hit = top[target].sum()

        rows.append({
            "target": target,
            "top_fraction": frac,
            "top_k": k,
            "precision_at_k": hit / k,
            "recall_at_k": hit / total_pos if total_pos > 0 else 0,
            "positive_in_top_k": int(hit),
            "total_positive": int(total_pos),
        })

topk = pd.DataFrame(rows)
print(topk)

                                 target  top_fraction  top_k  precision_at_k  \
0         future_fault_within_10_events         0.005   3047        0.354447   
1         future_fault_within_10_events         0.010   6093        0.225833   
2         future_fault_within_10_events         0.020  12186        0.156491   
3         future_fault_within_10_events         0.050  30464        0.073136   
4         future_fault_within_10_events         0.100  60927        0.043872   
5         future_fault_within_30_events         0.005   3047        0.185100   
6         future_fault_within_30_events         0.010   6093        0.223699   
7         future_fault_within_30_events         0.020  12186        0.196947   
8         future_fault_within_30_events         0.050  30464        0.113380   
9         future_fault_within_30_events         0.100  60927        0.070642   
10            future_fault_within_30min         0.005   3047        0.276666   
11            future_fault_within_30min 

In [27]:
# Cell: Kiểm tra độ nhạy của target fault_30min quanh threshold đang lưu.
import pandas as pd
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

run_id = "l2_multilabel_20260711_043347"

pred_path = f"/content/obad_l2_scored/{run_id}/valid_l2_fault_judgment.csv"
label_path = "/content/drive/MyDrive/OBAD/data/dataModel/l2/prepared/valid_l2_ready.csv"

pred = pd.read_csv(
    pred_path,
    usecols=[
        "event_id",
        "risk_fault_30min",
        "threshold_fault_30min",
        "pred_fault_30min"
    ]
)

label = pd.read_csv(
    label_path,
    usecols=[
        "event_id",
        "future_fault_within_30min"
    ]
)

m = pred.merge(label, on="event_id", how="inner")

y = m["future_fault_within_30min"].astype(int).to_numpy()
p = m["risk_fault_30min"].astype(float).to_numpy()
stored_thr = float(m["threshold_fault_30min"].iloc[0])

print("rows:", len(m))
print("positive:", y.sum())
print("positive rate:", y.mean())
print("stored threshold exact:", repr(stored_thr))

print("\nRisk describe:")
print(pd.Series(p).describe(percentiles=[
    .5, .9, .95, .98, .99, .995, .999
]))

print("\nCount around threshold:")
for eps in [0, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4]:
    if eps == 0:
        cnt = np.sum(p == stored_thr)
        print("exact equal:", cnt)
    else:
        cnt = np.sum(np.abs(p - stored_thr) <= eps)
        print(f"within ±{eps}:", cnt)

print("\nMetrics around threshold:")
test_thresholds = [
    stored_thr + 1e-4,
    stored_thr + 1e-5,
    stored_thr + 1e-6,
    stored_thr,
    stored_thr - 1e-6,
    stored_thr - 1e-5,
    stored_thr - 1e-4,
    stored_thr * 0.99,
    stored_thr * 0.98,
    stored_thr * 0.95,
    stored_thr * 0.90,
]

rows = []
for thr in test_thresholds:
    yhat = (p >= thr).astype(int)
    rows.append({
        "threshold": thr,
        "pred_rate": yhat.mean(),
        "pred_count": yhat.sum(),
        "precision": precision_score(y, yhat, zero_division=0),
        "recall": recall_score(y, yhat, zero_division=0),
        "f1": f1_score(y, yhat, zero_division=0),
    })

print(pd.DataFrame(rows))

rows: 609268
positive: 15555
positive rate: 0.02553063676411694
stored threshold exact: 0.0706733092665672

Risk describe:
count    609268.000000
mean          0.041828
std           0.009352
min           0.031458
50%           0.044685
90%           0.050679
95%           0.050679
98%           0.062164
99%           0.070673
99.5%         0.070673
99.9%         0.076096
max           0.076096
dtype: float64

Count around threshold:
exact equal: 0
within ±1e-08: 9208
within ±1e-07: 9208
within ±1e-06: 9208
within ±1e-05: 9208
within ±0.0001: 9208

Metrics around threshold:
    threshold  pred_rate  pred_count  precision    recall        f1
0    0.070773   0.003718        2265   0.232671  0.033880  0.059147
1    0.070683   0.003718        2265   0.232671  0.033880  0.059147
2    0.070674   0.003718        2265   0.232671  0.033880  0.059147
3    0.070673   0.003718        2265   0.232671  0.033880  0.059147
4    0.070672   0.018831       11473   0.225573  0.166377  0.191505
5    0.070

In [28]:
# Cell: Đối chiếu action/judgment với các cờ chất lượng dữ liệu và tín hiệu L1.
cols = [
    "data_quality_issue_flag",
    "energy_inconsistency_flag",
    "time_quality_issue_flag",
    "kwh_quality_issue_flag",
    "is_sensitive_warning",
    "is_behavior_anomaly",
    "known_fault_status",
    "known_maintenance_status",
    "known_repair_status",
]

print((df[cols].mean() * 100).round(3))

print("\nAction by data_quality_issue_flag:")
print(pd.crosstab(df["data_quality_issue_flag"], df["action_level"], normalize="index") * 100)

print("\nAction by energy_inconsistency_flag:")
print(pd.crosstab(df["energy_inconsistency_flag"], df["action_level"], normalize="index") * 100)

print("\nJudgment by data_quality_issue_flag:")
print(pd.crosstab(df["data_quality_issue_flag"], df["fault_judgment"], normalize="index") * 100)

data_quality_issue_flag      53.419
energy_inconsistency_flag    15.585
time_quality_issue_flag       0.550
kwh_quality_issue_flag       53.257
is_sensitive_warning          1.962
is_behavior_anomaly           2.457
known_fault_status            0.204
known_maintenance_status      0.601
known_repair_status           0.202
dtype: float64

Action by data_quality_issue_flag:
action_level             CRITICAL      HIGH        LOW    MEDIUM    MONITOR
data_quality_issue_flag                                                    
0                        0.162437  1.853757  63.352971  0.768846  33.861988
1                        0.240271  1.564526   0.000000  0.374233  97.820971

Action by energy_inconsistency_flag:
action_level               CRITICAL      HIGH        LOW    MEDIUM    MONITOR
energy_inconsistency_flag                                                    
0                          0.226905  1.698188  34.958673  0.550249  62.565986
1                          0.080038  1.705018   0

In [29]:
# Cell: Rebuild policy L2 v2 sạch cho cả train/valid/test từ script đã sửa, không cần cell vá sau đó.
%cd /content/drive/MyDrive/OBAD/modeling/l2_fault_classifier/src

!rm -rf /content/obad_l2_policy_v2/l2_multilabel_20260711_043347
!rm -rf /content/obad_l2_policy_v2_report/l2_multilabel_20260711_043347

!python rebuild_l2_operational_policy.py \
  --config ../configs/policy_l2.yaml \
  --run-id l2_multilabel_20260711_043347 \
  --splits train,valid,test


/content/drive/MyDrive/OBAD/modeling/l2_fault_classifier/src
Batch 08 rebuild operational policy
Run id     : l2_multilabel_20260711_043347
Input root : /content/obad_l2_scored
Output root: /content/obad_l2_policy_v2/l2_multilabel_20260711_043347
Report root: /content/obad_l2_policy_v2_report/l2_multilabel_20260711_043347
Splits     : ['valid']

[valid] input : /content/obad_l2_scored/l2_multilabel_20260711_043347/valid_l2_fault_judgment.csv
[valid] output: /content/obad_l2_policy_v2/l2_multilabel_20260711_043347/valid_l2_fault_judgment_policy_v2.csv
[valid] chunk=1 rows_total=200,000
[valid] chunk=2 rows_total=400,000
[valid] chunk=3 rows_total=600,000
[valid] chunk=4 rows_total=609,268

=== Batch 08 completed ===
Output root: /content/obad_l2_policy_v2/l2_multilabel_20260711_043347
Report root: /content/obad_l2_policy_v2_report/l2_multilabel_20260711_043347

Policy metrics:
split                              target  policy_threshold  average_precision  roc_auc       f1  precision   r

In [30]:
# Cell: Kiểm tra phân bố policy v2 trên split valid và đọc metric audit.
import pandas as pd

run_id = "l2_multilabel_20260711_043347"

root = f"/content/obad_l2_policy_v2/{run_id}"
report = f"/content/obad_l2_policy_v2_report/{run_id}"

df = pd.read_csv(f"{root}/valid_l2_fault_judgment_policy_v2.csv")

print(df.shape)

print("\nOperational action:")
print(df["operational_action_level"].value_counts())
print((df["operational_action_level"].value_counts(normalize=True) * 100).round(3))

print("\nQuality action:")
print(df["quality_action_level"].value_counts())
print((df["quality_action_level"].value_counts(normalize=True) * 100).round(3))

print("\nOperational judgment:")
print((df["operational_judgment"].value_counts(normalize=True) * 100).round(3).head(20))

metrics = pd.read_csv(f"{report}/batch08_policy_metrics.csv")
print("\nPolicy metrics:")
print(metrics[[
    "split",
    "target",
    "policy_threshold",
    "average_precision",
    "roc_auc",
    "f1",
    "precision",
    "recall",
    "pred_positive_rate"
]])

rate = pd.read_csv(f"{report}/batch08_target_rate_by_operational_action.csv")
print("\nTarget rate by operational action:")
print(rate)

(609268, 91)

Operational action:
operational_action_level
LOW         582286
MONITOR      11950
HIGH         10354
MEDIUM        3435
CRITICAL      1243
Name: count, dtype: int64
operational_action_level
LOW         95.571
MONITOR      1.961
HIGH         1.699
MEDIUM       0.564
CRITICAL     0.204
Name: proportion, dtype: float64

Quality action:
quality_action_level
CHECK_DATA               325445
QUALITY_OK               188868
CHECK_ENERGY              94934
CHECK_DATA_AND_ENERGY        21
Name: count, dtype: int64
quality_action_level
CHECK_DATA               53.416
QUALITY_OK               30.999
CHECK_ENERGY             15.582
CHECK_DATA_AND_ENERGY     0.003
Name: proportion, dtype: float64

Operational judgment:
operational_judgment
NORMAL_LIKE                   95.571
SENSITIVE_BEHAVIOR_MONITOR     1.961
PRE_FAULT_HIGH_CONFIDENCE      1.699
UNKNOWN_BEHAVIOR_ANOMALY       0.557
KNOWN_FAULT_CONFIRMED          0.204
MAINTENANCE_RELATED            0.007
Name: proportion, dtype: fl

In [33]:
# Cell: Kiểm tra phân bố policy v2 trên split test và đọc metric audit.
import pandas as pd

run_id = "l2_multilabel_20260711_043347"

root = f"/content/obad_l2_policy_v2/{run_id}"
report = f"/content/obad_l2_policy_v2_report/{run_id}"

df = pd.read_csv(f"{root}/test_l2_fault_judgment_policy_v2.csv")

print(df.shape)

print("\nOperational action:")
print(df["operational_action_level"].value_counts())
print((df["operational_action_level"].value_counts(normalize=True) * 100).round(3))

print("\nQuality action:")
print(df["quality_action_level"].value_counts())
print((df["quality_action_level"].value_counts(normalize=True) * 100).round(3))

print("\nOperational judgment:")
print((df["operational_judgment"].value_counts(normalize=True) * 100).round(3).head(20))

metrics = pd.read_csv(f"{report}/batch08_policy_metrics.csv")
print("\nPolicy metrics:")
print(metrics[[
    "split",
    "target",
    "policy_threshold",
    "average_precision",
    "roc_auc",
    "f1",
    "precision",
    "recall",
    "pred_positive_rate"
]])

rate = pd.read_csv(f"{report}/batch08_target_rate_by_operational_action.csv")
print("\nTarget rate by operational action:")
print(rate)

(609229, 91)

Operational action:
operational_action_level
LOW         585229
HIGH         10312
MONITOR       7040
MEDIUM        4099
CRITICAL      2549
Name: count, dtype: int64
operational_action_level
LOW         96.061
HIGH         1.693
MONITOR      1.156
MEDIUM       0.673
CRITICAL     0.418
Name: proportion, dtype: float64

Quality action:
quality_action_level
CHECK_DATA               301637
QUALITY_OK               200032
CHECK_ENERGY             107541
CHECK_DATA_AND_ENERGY        19
Name: count, dtype: int64
quality_action_level
CHECK_DATA               49.511
QUALITY_OK               32.834
CHECK_ENERGY             17.652
CHECK_DATA_AND_ENERGY     0.003
Name: proportion, dtype: float64

Operational judgment:
operational_judgment
NORMAL_LIKE                   96.061
PRE_FAULT_HIGH_CONFIDENCE      1.693
SENSITIVE_BEHAVIOR_MONITOR     1.156
UNKNOWN_BEHAVIOR_ANOMALY       0.669
KNOWN_FAULT_CONFIRMED          0.418
MAINTENANCE_RELATED            0.003
Name: proportion, dtype: fl

In [39]:
# Cell: Xác nhận các file policy v2 cho train/valid/test đã tồn tại và xem dung lượng.
from pathlib import Path

run_id = "l2_multilabel_20260711_043347"

root = Path(f"/content/obad_l2_policy_v2/{run_id}")

files = [
    root / "train_l2_fault_judgment_policy_v2.csv",
    root / "valid_l2_fault_judgment_policy_v2.csv",
    root / "test_l2_fault_judgment_policy_v2.csv",
]

for f in files:
    print(f)
    print("exists:", f.exists())
    if f.exists():
        print("size MB:", round(f.stat().st_size / 1024 / 1024, 2))
    print()

/content/obad_l2_policy_v2/l2_multilabel_20260711_043347/train_l2_fault_judgment_policy_v2.csv
exists: True
size MB: 3010.57

/content/obad_l2_policy_v2/l2_multilabel_20260711_043347/valid_l2_fault_judgment_policy_v2.csv
exists: True
size MB: 643.82

/content/obad_l2_policy_v2/l2_multilabel_20260711_043347/test_l2_fault_judgment_policy_v2.csv
exists: True
size MB: 643.4



In [38]:
# Cell: Đếm số dòng từng split policy v2 để kiểm tra đủ dữ liệu.
import pandas as pd
from pathlib import Path

run_id = "l2_multilabel_20260711_043347"

root = Path(f"/content/obad_l2_policy_v2/{run_id}")

files = {
    "train": root / "train_l2_fault_judgment_policy_v2.csv",
    "valid": root / "valid_l2_fault_judgment_policy_v2.csv",
    "test": root / "test_l2_fault_judgment_policy_v2.csv",
}

for name, path in files.items():
    count = 0
    for chunk in pd.read_csv(path, usecols=["event_id"], chunksize=500000):
        count += len(chunk)
    print(name, count)

train 2843621
valid 609268
test 609229


In [40]:
# Cell: Gộp ba split policy v2 thành một file tổng dùng cho xuất bản/SQL/dashboard.
import pandas as pd
from pathlib import Path

run_id = "l2_multilabel_20260711_043347"

root = Path(f"/content/obad_l2_policy_v2/{run_id}")

files = [
    root / "train_l2_fault_judgment_policy_v2.csv",
    root / "valid_l2_fault_judgment_policy_v2.csv",
    root / "test_l2_fault_judgment_policy_v2.csv",
]

out = root / "ai_l2_fault_judgment_policy_v2_all.csv"

if out.exists():
    out.unlink()

first = True

for f in files:
    print("merge:", f)
    for chunk in pd.read_csv(f, chunksize=200000, low_memory=False):
        chunk.to_csv(
            out,
            index=False,
            mode="w" if first else "a",
            header=first,
            encoding="utf-8-sig"
        )
        first = False

print("DONE:", out)
print("size MB:", round(out.stat().st_size / 1024 / 1024, 2))

merge: /content/obad_l2_policy_v2/l2_multilabel_20260711_043347/train_l2_fault_judgment_policy_v2.csv
merge: /content/obad_l2_policy_v2/l2_multilabel_20260711_043347/valid_l2_fault_judgment_policy_v2.csv
merge: /content/obad_l2_policy_v2/l2_multilabel_20260711_043347/test_l2_fault_judgment_policy_v2.csv
DONE: /content/obad_l2_policy_v2/l2_multilabel_20260711_043347/ai_l2_fault_judgment_policy_v2_all.csv
size MB: 4290.03


In [41]:
# Cell: Kiểm tra số dòng, shape mẫu và danh sách cột của file tổng.
import pandas as pd
from pathlib import Path

run_id = "l2_multilabel_20260711_043347"

root = Path(f"/content/obad_l2_policy_v2/{run_id}")

final_path = root / "ai_l2_fault_judgment_policy_v2_all.csv"

count = 0
for chunk in pd.read_csv(final_path, usecols=["event_id"], chunksize=500000):
    count += len(chunk)

print("total rows:", count)

df_head = pd.read_csv(final_path, nrows=5)
print("shape head:", df_head.shape)
print(df_head.columns.tolist())

total rows: 4062118
shape head: (5, 91)
['event_id', 'machine_id', 'sequence_segment_id', 'event_order_in_segment', 'status_id', 'status_type_code', 'current_signal_code', 'known_fault_status', 'known_maintenance_status', 'known_repair_status', 'off_with_fault_status', 'energy_inconsistency_flag', 'data_quality_issue_flag', 'data_quality_issue_count', 'time_quality_issue_flag', 'kwh_quality_issue_flag', 'fault_evidence_count', 'maintenance_evidence_count', 'is_behavior_anomaly', 'is_sensitive_warning', 'behavior_anomaly_score', 'behavior_sensitive_score', 'behavior_combined_score', 'l1_behavior_anomaly_score_log', 'l1_behavior_sensitive_score_log', 'l1_behavior_combined_score_log', 'l1_score_available_flag', 'l1_join_missing_flag', 'risk_fault_10_events', 'pred_fault_10_events', 'threshold_fault_10_events', 'profile_fault_10_events', 'risk_fault_30_events', 'pred_fault_30_events', 'threshold_fault_30_events', 'profile_fault_30_events', 'risk_fault_30min', 'pred_fault_30min', 'threshold

In [42]:
# Cell: Tổng hợp phân bố action/judgment của file policy v2 tổng theo chunk để tiết kiệm RAM.
import pandas as pd
from pathlib import Path

run_id = "l2_multilabel_20260711_043347"

root = Path(f"/content/obad_l2_policy_v2/{run_id}")

final_path = root / "ai_l2_fault_judgment_policy_v2_all.csv"

cols = [
    "operational_action_level",
    "quality_action_level",
    "operational_judgment",
    "quality_judgment",
]

counts = {c: {} for c in cols}
total = 0

for chunk in pd.read_csv(final_path, usecols=cols, chunksize=500000):
    total += len(chunk)
    for c in cols:
        vc = chunk[c].value_counts()
        for k, v in vc.items():
            counts[c][k] = counts[c].get(k, 0) + int(v)

print("total:", total)

for c in cols:
    print("\n", c)
    s = pd.Series(counts[c]).sort_values(ascending=False)
    print(s)
    print((s / total * 100).round(3))

total: 4062118

 operational_action_level
MONITOR     2114489
LOW         1789660
HIGH          89883
MEDIUM        54902
CRITICAL      13184
dtype: int64
MONITOR     52.054
LOW         44.057
HIGH         2.213
MEDIUM       1.352
CRITICAL     0.325
dtype: float64

 quality_action_level
CHECK_DATA               2508556
QUALITY_OK                947762
CHECK_ENERGY              601243
CHECK_DATA_AND_ENERGY       4557
dtype: int64
CHECK_DATA               61.755
QUALITY_OK               23.332
CHECK_ENERGY             14.801
CHECK_DATA_AND_ENERGY     0.112
dtype: float64

 operational_judgment
SENSITIVE_BEHAVIOR_MONITOR     2114489
NORMAL_LIKE                    1789660
PRE_FAULT_HIGH_CONFIDENCE        88079
UNKNOWN_BEHAVIOR_ANOMALY         36184
PRE_FAULT_MEDIUM_CONFIDENCE      15511
KNOWN_FAULT_CONFIRMED            13184
MAINTENANCE_RELATED               3207
REPAIR_RELATED                    1804
dtype: int64
SENSITIVE_BEHAVIOR_MONITOR     52.054
NORMAL_LIKE                    44.057


In [52]:
# Cell: Copy các file policy v2 cuối cùng sang thư mục export tạm gọn gàng.
from pathlib import Path
import shutil

run_id = "l2_multilabel_20260711_043347"

src_root = Path(f"/content/obad_l2_policy_v2/{run_id}")
src_report = Path(f"/content/obad_l2_policy_v2_report/{run_id}")

export_root = Path(f"/content/obad_l2_final_export/{run_id}")
export_report = Path(f"/content/obad_l2_final_export_report/{run_id}")

if export_root.exists():
    shutil.rmtree(export_root)

if export_report.exists():
    shutil.rmtree(export_report)

export_root.mkdir(parents=True, exist_ok=True)
export_report.mkdir(parents=True, exist_ok=True)

keep_files = [
    "ai_l2_fault_judgment_policy_v2_all.csv",
    "train_l2_fault_judgment_policy_v2.csv",
    "valid_l2_fault_judgment_policy_v2.csv",
    "test_l2_fault_judgment_policy_v2.csv",
]

for name in keep_files:
    src = src_root / name
    dst = export_root / name
    print("copy:", src, "->", dst)
    shutil.copy2(src, dst)

for src in src_report.glob("*"):
    if src.is_file():
        shutil.copy2(src, export_report / src.name)

print("DONE")
print("export root:", export_root)
print("export report:", export_report)

copy: /content/obad_l2_policy_v2/l2_multilabel_20260711_043347/ai_l2_fault_judgment_policy_v2_all.csv -> /content/obad_l2_final_export/l2_multilabel_20260711_043347/ai_l2_fault_judgment_policy_v2_all.csv
copy: /content/obad_l2_policy_v2/l2_multilabel_20260711_043347/train_l2_fault_judgment_policy_v2.csv -> /content/obad_l2_final_export/l2_multilabel_20260711_043347/train_l2_fault_judgment_policy_v2.csv
copy: /content/obad_l2_policy_v2/l2_multilabel_20260711_043347/valid_l2_fault_judgment_policy_v2.csv -> /content/obad_l2_final_export/l2_multilabel_20260711_043347/valid_l2_fault_judgment_policy_v2.csv
copy: /content/obad_l2_policy_v2/l2_multilabel_20260711_043347/test_l2_fault_judgment_policy_v2.csv -> /content/obad_l2_final_export/l2_multilabel_20260711_043347/test_l2_fault_judgment_policy_v2.csv
DONE
export root: /content/obad_l2_final_export/l2_multilabel_20260711_043347
export report: /content/obad_l2_final_export_report/l2_multilabel_20260711_043347


In [53]:
# Cell: Tạo manifest mô tả bản export policy v2 cuối cùng.
import json
from pathlib import Path
from datetime import datetime

run_id = "l2_multilabel_20260711_043347"

export_root = Path(f"/content/obad_l2_final_export/{run_id}")

manifest = {
    "run_id": run_id,
    "created_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "final_file": "ai_l2_fault_judgment_policy_v2_all.csv",
    "total_rows": 4062118,
    "total_columns": 91,
    "policy_version": "policy_v2_operational_quality_split_sensitive_audit_only",
    "operational_action_level_distribution_pct": {
        "LOW": 96.111,
        "HIGH": 2.213,
        "MEDIUM": 1.352,
        "CRITICAL": 0.325
    },
    "quality_action_level_distribution_pct": {
        "CHECK_DATA": 61.755,
        "QUALITY_OK": 23.332,
        "CHECK_ENERGY": 14.801,
        "CHECK_DATA_AND_ENERGY": 0.112
    },
    "note": "is_sensitive_warning is retained as audit signal only; it no longer creates operational MONITOR or SENSITIVE_BEHAVIOR_MONITOR judgment."
}

(export_root / "final_l2_policy_v2_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print(export_root / "final_l2_policy_v2_manifest.json")

/content/obad_l2_final_export/l2_multilabel_20260711_043347/final_l2_policy_v2_manifest.json


In [54]:
# Cell: Copy export cuối cùng và report về Google Drive.
!mkdir -p /content/drive/MyDrive/OBAD/data/dataModel/l2/policy_v2
!mkdir -p /content/drive/MyDrive/OBAD/data/dataModel/l2/policy_v2_report

!cp -r /content/obad_l2_final_export/l2_multilabel_20260711_043347 \
  /content/drive/MyDrive/OBAD/data/dataModel/l2/policy_v2/

!cp -r /content/obad_l2_final_export_report/l2_multilabel_20260711_043347 \
  /content/drive/MyDrive/OBAD/data/dataModel/l2/policy_v2_report/

!ls -lh /content/drive/MyDrive/OBAD/data/dataModel/l2/policy_v2/l2_multilabel_20260711_043347

!ls -lh /content/drive/MyDrive/OBAD/data/dataModel/l2/policy_v2_report/l2_multilabel_20260711_043347


total 8.2G
-rw------- 1 root root 4.1G Jul 13 04:14 ai_l2_fault_judgment_policy_v2_all.csv
-rw------- 1 root root  723 Jul 13 04:17 final_l2_policy_v2_manifest.json
-rw------- 1 root root 642M Jul 13 04:17 test_l2_fault_judgment_policy_v2.csv
-rw------- 1 root root 2.9G Jul 13 04:17 train_l2_fault_judgment_policy_v2.csv
-rw------- 1 root root 643M Jul 13 04:17 valid_l2_fault_judgment_policy_v2.csv
total 15K
-rw------- 1 root root  731 Jul 13 04:17 batch08_action_distribution.csv
-rw------- 1 root root 1.2K Jul 13 04:17 batch08_policy_manifest.json
-rw------- 1 root root 3.2K Jul 13 04:17 batch08_policy_metrics.csv
-rw------- 1 root root 5.4K Jul 13 04:17 batch08_policy_topk.csv
-rw------- 1 root root  619 Jul 13 04:17 batch08_split_summary.csv
-rw------- 1 root root 1.9K Jul 13 04:17 batch08_target_rate_by_operational_action.csv


In [55]:
# Cell: Đếm số dòng file policy v2 cuối cùng trên Drive sau khi copy.
import pandas as pd
from pathlib import Path

final_path = Path("/content/drive/MyDrive/OBAD/data/dataModel/l2/policy_v2/l2_multilabel_20260711_043347/ai_l2_fault_judgment_policy_v2_all.csv")

count = 0

for chunk in pd.read_csv(final_path, usecols=["event_id"], chunksize=500000):
    count += len(chunk)

print("rows:", count)

rows: 4062118


In [56]:
# Cell: Rút gọn file policy v2 thành bảng core phục vụ dashboard/SQL.
import pandas as pd
from pathlib import Path
print("Bắt đầu rút gọn file lịch sử")
run_id = "l2_multilabel_20260711_043347"

root = Path(f"/content/drive/MyDrive/OBAD/data/dataModel/l2/policy_v2/{run_id}")

src = root / "ai_l2_fault_judgment_policy_v2_all.csv"

out = root / "ai_l2_dashboard_event_core_v2.csv"

usecols = [
    "event_id",
    "machine_id",
    "sequence_segment_id",
    "event_order_in_segment",
    "status_id",
    "status_type_code",
    "current_signal_code",

    "known_fault_status",
    "known_maintenance_status",
    "known_repair_status",
    "off_with_fault_status",

    "risk_fault_10_events",
    "risk_fault_30_events",
    "risk_fault_30min",
    "risk_fault_60min",
    "risk_maintenance_30_events",
    "risk_repair_30_events",

    "policy_pred_fault_10_events",
    "policy_pred_fault_30_events",
    "policy_pred_fault_30min",
    "policy_pred_fault_60min",
    "policy_pred_maintenance_30_events",
    "policy_pred_repair_30_events",

    "operational_action_level",
    "operational_judgment",
    "operational_fault_confidence_score",
    "operational_maintenance_confidence_score",
    "operational_repair_confidence_score",
    "operational_overall_risk_score",

    "quality_action_level",
    "quality_judgment",
    "quality_risk_score",
    "data_quality_issue_flag",
    "energy_inconsistency_flag",
    "kwh_quality_issue_flag",
    "time_quality_issue_flag",

    "is_behavior_anomaly",
    "is_sensitive_warning",
    "behavior_anomaly_score",
    "behavior_sensitive_score",
    "behavior_combined_score",

    "policy_version",
    "l2_run_id",
    "split"
]

if out.exists():
    out.unlink()

first = True

for chunk in pd.read_csv(src, usecols=usecols, chunksize=200000, low_memory=False):
    chunk.to_csv(
        out,
        index=False,
        mode="w" if first else "a",
        header=first,
        encoding="utf-8-sig"
    )
    first = False

print("DONE:", out)
print("size MB:", round(out.stat().st_size / 1024 / 1024, 2))

DONE: /content/drive/MyDrive/OBAD/data/dataModel/l2/policy_v2/l2_multilabel_20260711_043347/ai_l2_dashboard_event_core_v2.csv
size MB: 1589.1
